In [1]:
from pathlib import Path
import hashlib
import pandas as pd

INPUT_DIR = Path("/kaggle/input")
RESULTS_DIR = Path("/kaggle/working/artifacts/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def find_one(filename):
    matches = list(INPUT_DIR.rglob(filename))

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one {filename}, found {len(matches)}: {matches}"
        )

    return matches[0]


# Verify the original dataset matches the previously audited labels.
dataset_dir = find_one("train.txt").parent

expected_hashes = {
    "train.txt": (
        "52af9a4fdc769396bc5ce530872cf0d51bce"
        "8fd7e25579b333007d4ebee6a6ba"
    ),
    "test.txt": (
        "7c92ae0bef49652005e1541a1f2b774e31"
        "3be4431a891cb0aa5539d37ca2a71e"
    ),
}

for filename, expected in expected_hashes.items():
    content = (dataset_dir / filename).read_bytes()
    actual = hashlib.sha256(content).hexdigest()

    assert actual == expected, f"Label checksum mismatch: {filename}"

# Load saved splits while preserving patient identifiers as strings.
training = pd.read_csv(
    find_one("train.csv"),
    dtype=str,
    keep_default_na=False,
)
validation = pd.read_csv(
    find_one("validation.csv"),
    dtype=str,
    keep_default_na=False,
)

required_columns = {
    "filename", "label", "patient_id",
    "source", "sha256", "split_group",
}

for name, table, expected_count in [
    ("Training", training, 23873),
    ("Validation", validation, 6103),
]:
    assert required_columns.issubset(table.columns)
    assert len(table) == expected_count
    assert table["filename"].is_unique
    assert table["sha256"].is_unique
    assert set(table["label"]) == {"negative", "positive"}

    missing = [
        filename
        for filename in table["filename"]
        if not (dataset_dir / "train" / filename).is_file()
    ]

    assert not missing, f"{name}: missing images: {missing[:5]}"
    print(f"{name} images: {len(table)}")

for column in ["filename", "sha256", "split_group"]:
    assert set(training[column]).isdisjoint(validation[column]), (
        f"Train/validation overlap: {column}"
    )

print("Dataset label checksums match.")
print("Saved split and image-path checks passed.")

Training images: 23873
Validation images: 6103
Dataset label checksums match.
Saved split and image-path checks passed.


In [6]:
import sys
import inspect
import importlib
from pathlib import Path

# Locate the updated attached source.
module_files = list(
    Path("/kaggle/input").glob("**/src/covid_xray/data.py")
)

if len(module_files) != 1:
    raise RuntimeError(
        f"Expected one preprocessing module, found: {module_files}"
    )

module_path = module_files[0].resolve()
source_dir = module_path.parents[1]

sys.path.insert(0, str(source_dir))
importlib.invalidate_caches()

# Remove cached project modules before importing the updated source.
for name in list(sys.modules):
    if name == "covid_xray" or name.startswith("covid_xray."):
        del sys.modules[name]

from covid_xray.data import preprocess_image, make_dataset, LABEL_MAP

assert Path(inspect.getfile(make_dataset)).resolve() == module_path

print("Updated shared source loaded.")
print("Batch loader:", make_dataset.__name__)
print("Class mapping:", LABEL_MAP)

Updated shared source loaded.
Batch loader: make_dataset
Class mapping: {'negative': 0, 'positive': 1}


In [3]:
import numpy as np
import tensorflow as tf

print("TensorFlow:", tf.__version__)

# Verify CPU execution before building the input pipeline.
with tf.device("/CPU:0"):
    result = tf.reduce_sum(tf.constant([1.0, 2.0, 3.0]))

assert float(result.numpy()) == 6.0
print("TensorFlow CPU check passed.")

TensorFlow: 2.20.0
TensorFlow CPU check passed.


2026-10-18 11:10:40.463032: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [8]:
BATCH_SIZE = 32
SEED = 42

# Use the shared batch loader from the project source.
train_ds = make_dataset(
    training,
    image_dir=dataset_dir / "train",
    batch_size=BATCH_SIZE,
    training_mode=True,
    seed=SEED,
)

val_ds = make_dataset(
    validation,
    image_dir=dataset_dir / "train",
    batch_size=BATCH_SIZE,
    training_mode=False,
)

images, labels = next(iter(train_ds))

tf.debugging.assert_shapes([
    (images, (32, 224, 224, 1)),
    (labels, (32, 1)),
])
tf.debugging.assert_all_finite(images, "Non-finite image pixels")
tf.debugging.assert_greater_equal(images, 0.0)
tf.debugging.assert_less_equal(images, 1.0)
tf.debugging.assert_equal(
    tf.reduce_all((labels == 0.0) | (labels == 1.0)),
    True,
)

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Training batches:", int(tf.data.experimental.cardinality(train_ds)))
print("Validation batches:", int(tf.data.experimental.cardinality(val_ds)))
print("Shared training batch check passed.")

Image batch shape: (32, 224, 224, 1)
Label batch shape: (32, 1)
Training batches: 747
Validation batches: 191
Shared training batch check passed.


In [9]:
val_images, val_labels = next(iter(val_ds))

tf.debugging.assert_shapes([
    (val_images, (32, 224, 224, 1)),
    (val_labels, (32, 1)),
])
tf.debugging.assert_all_finite(
    val_images, "Non-finite validation pixels"
)
tf.debugging.assert_greater_equal(val_images, 0.0)
tf.debugging.assert_less_equal(val_images, 1.0)
tf.debugging.assert_equal(
    tf.reduce_all((val_labels == 0.0) | (val_labels == 1.0)),
    True,
)

# Verify validation labels preserve manifest order.
expected_labels = (
    validation["label"]
    .map(LABEL_MAP)
    .iloc[:32]
    .to_numpy(dtype=np.float32)
    .reshape(-1, 1)
)

np.testing.assert_array_equal(
    val_labels.numpy(),
    expected_labels,
)

print("Validation image shape:", val_images.shape)
print("Validation label shape:", val_labels.shape)
print("Validation batch and label-order checks passed.")

Validation image shape: (32, 224, 224, 1)
Validation label shape: (32, 1)
Validation batch and label-order checks passed.


In [10]:
import time

pipeline_checks = []

for split_name, dataset, expected_count in [
    ("train", train_ds, len(training)),
    ("validation", val_ds, len(validation)),
]:
    start = time.perf_counter()
    image_count = 0
    batch_count = 0
    positive_count = 0

    for batch_images, batch_labels in dataset:
        tf.debugging.assert_all_finite(
            batch_images, "Non-finite image pixels"
        )
        tf.debugging.assert_greater_equal(batch_images, 0.0)
        tf.debugging.assert_less_equal(batch_images, 1.0)
        tf.debugging.assert_equal(
            tf.shape(batch_images)[1:], [224, 224, 1]
        )
        tf.debugging.assert_equal(
            tf.shape(batch_labels),
            [tf.shape(batch_images)[0], 1],
        )
        tf.debugging.assert_equal(
            tf.reduce_all(
                (batch_labels == 0.0) | (batch_labels == 1.0)
            ),
            True,
        )

        image_count += int(tf.shape(batch_images)[0].numpy())
        positive_count += int(tf.reduce_sum(batch_labels).numpy())
        batch_count += 1

        if batch_count % 100 == 0:
            print(f"{split_name}: {image_count} images checked")

    expected_table = (
        training if split_name == "train" else validation
    )
    expected_positive = int(
        (expected_table["label"] == "positive").sum()
    )

    assert image_count == expected_count
    assert positive_count == expected_positive

    elapsed = time.perf_counter() - start

    pipeline_checks.append({
        "split": split_name,
        "images": image_count,
        "batches": batch_count,
        "positive": positive_count,
        "seconds": round(elapsed, 2),
    })

    print(
        f"{split_name}: PASS | {image_count} images | "
        f"{batch_count} batches | {elapsed:.1f} seconds"
    )

pipeline_report = pd.DataFrame(pipeline_checks)
pipeline_report.to_csv(
    RESULTS_DIR / "pipeline_validation.csv",
    index=False,
)

print("\nFull pipeline checks passed.")

train: 3200 images checked
train: 6400 images checked
train: 9600 images checked
train: 12800 images checked
train: 16000 images checked
train: 19200 images checked
train: 22400 images checked
train: PASS | 23873 images | 747 batches | 331.2 seconds
validation: 3200 images checked
validation: PASS | 6103 images | 191 batches | 94.8 seconds

Full pipeline checks passed.


## Preprocessing and Batch Pipeline

The notebook imports the shared preprocessing function from
the attached project source.

Images are converted to grayscale, resized with aspect-ratio
preservation and black padding, and normalized to [0, 1].
Output shape is (224, 224, 1), with float32 pixels.

Class mapping:
- negative: 0
- positive: 1

The pipeline uses batches of 32:
- Training: 23,873 images, 747 batches.
- Validation: 6,103 images, 191 batches.

Training paths are shuffled before decoding. Validation order
is preserved. The final batch is retained even when incomplete.

One actual batch from each split passed shape, finite-value,
pixel-range, and binary-label checks. Validation labels also
matched manifest order.

Full training and validation pipeline iteration completed successfully
for all 29,976 images. Image shapes, finite pixel values, pixel ranges,
binary labels, image counts, and positive-label counts were verified.

CPU execution took approximately 331 seconds for training data and
95 seconds for validation data in this session. These timings exclude
model training.

Model training has not yet been performed.